# 03 - Heatmaps And Report Artifact Explorer

This notebook recreates compact versions of the interaction heatmaps in `outputs/reports/metric_analysis/rendered/metric_analysis.tex` and inspects the generated report artifacts.

Use small grids while exploring. Increase the grid size only when you need report-quality detail.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import json


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "baseline.yaml").exists() and (
            candidate / "simulation" / "config_loader.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import result_metrics_from_result
from analysis.plot_style import driver_heatmap_cmap
from simulation.config_loader import load_config
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import ThresholdRule

BASE_CONFIG = load_config(REPO_DIR / "configs" / "baseline.yaml")
pd.options.display.max_columns = 120


## Heatmap Helper

The plotting helper is reused below; the scenario changes stay inline in each grid cell.


In [ ]:
def draw_heatmap(df, x_name, y_name, metric, title, driver, diverging=False):
    table = df.pivot(index=y_name, columns=x_name, values=metric).sort_index()
    fig, ax = plt.subplots(figsize=(7, 5))
    cmap = driver_heatmap_cmap(driver, diverging=diverging)
    norm = None
    if diverging:
        max_abs = float(np.nanmax(np.abs(table.values)))
        norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs) if max_abs else None
    image = ax.imshow(table.values, origin="lower", aspect="auto", cmap=cmap, norm=norm)
    ax.set_title(title)
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels([f"{v:.2g}" for v in table.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels([f"{v:.2g}" for v in table.index])
    fig.colorbar(image, ax=ax, shrink=0.85)
    fig.tight_layout()
    return fig


## Balking Step Interaction

This compact grid mirrors the report's balking-step heatmaps.

In [ ]:
STEP_VALUES = np.linspace(0.0, 1.0, 6)
base_balk_rules = {
    class_id: params.balk_prob
    for class_id, params in BASE_CONFIG.classes.items()
}

rows = []
for class_1_step in STEP_VALUES:
    for class_2_step in STEP_VALUES:
        class_1_rule = base_balk_rules[1]
        class_2_rule = base_balk_rules[2]
        classes = {
            **BASE_CONFIG.classes,
            1: replace(
                BASE_CONFIG.classes[1],
                balk_prob=ThresholdRule(
                    threshold=class_1_rule.threshold,
                    low=class_1_rule.low,
                    high=min(class_1_rule.low + float(class_1_step), 1.0),
                ),
            ),
            2: replace(
                BASE_CONFIG.classes[2],
                balk_prob=ThresholdRule(
                    threshold=class_2_rule.threshold,
                    low=class_2_rule.low,
                    high=min(class_2_rule.low + float(class_2_step), 1.0),
                ),
            ),
        }
        config = replace(BASE_CONFIG, classes=classes, seed=2027)
        result = ClinicAppointmentSimulation(config).run()
        rows.append(
            {
                "class_1_balk_step": class_1_step,
                "class_2_balk_step": class_2_step,
                **result_metrics_from_result(result),
            }
        )

balk_grid = pd.DataFrame(rows)
display(balk_grid.head())

draw_heatmap(balk_grid, "class_1_balk_step", "class_2_balk_step", "overall_percent_serviced", "Overall served rate", "balking")
draw_heatmap(balk_grid, "class_1_balk_step", "class_2_balk_step", "access_advantage_class_1", "Class 1 access advantage", "balking", diverging=True)


## Cancellation Interaction

Cancellation has one of the strongest class-gap effects in the report.

In [ ]:
CANCEL_VALUES = np.linspace(0.0, 0.30, 6)

rows = []
for class_1_cancel in CANCEL_VALUES:
    for class_2_cancel in CANCEL_VALUES:
        classes = {
            **BASE_CONFIG.classes,
            1: replace(BASE_CONFIG.classes[1], cancel_prob=float(class_1_cancel)),
            2: replace(BASE_CONFIG.classes[2], cancel_prob=float(class_2_cancel)),
        }
        config = replace(BASE_CONFIG, classes=classes, seed=2027)
        result = ClinicAppointmentSimulation(config).run()
        rows.append(
            {
                "class_1_cancel_prob": class_1_cancel,
                "class_2_cancel_prob": class_2_cancel,
                **result_metrics_from_result(result),
            }
        )

cancel_grid = pd.DataFrame(rows)

draw_heatmap(cancel_grid, "class_1_cancel_prob", "class_2_cancel_prob", "overall_percent_serviced", "Overall served rate", "cancellation")
draw_heatmap(cancel_grid, "class_1_cancel_prob", "class_2_cancel_prob", "access_advantage_class_1", "Class 1 access advantage", "cancellation", diverging=True)


## Inspect Generated Report Artifacts

The manifest is written by `scripts/generate_metric_analysis_figures.py` and records the command, config hashes, package versions, seeds, grids, row counts, and artifacts generated in that run.

In [ ]:
manifest_path = REPO_DIR / "outputs" / "reports" / "metric_analysis" / "manifest.json"
manifest = json.loads(manifest_path.read_text())

print("command:", " ".join(manifest["command"]))
print("git:", manifest["git"])
print("generated artifacts:", len(manifest["generated_artifacts"]))

display(pd.Series(manifest["row_counts"], name="rows").to_frame())
display(pd.Series(manifest["environment"]["packages"], name="version").to_frame())

## Open Existing Report Figures

Use this cell to inspect any generated figure without leaving the notebook.
Change `figure_name` to another file listed in the manifest.

In [ ]:
from IPython.display import Image, display

figure_name = "metric_access_drivers.png"
figure_path = REPO_DIR / "outputs" / "reports" / "metric_analysis" / "figures" / figure_name
print(figure_path)
display(Image(filename=str(figure_path)))